# Lesson 3.7 — Merge and Join Operations

**Objectives**
- Explain the different join types (inner, left, right, outer) using plain examples
- Combine two DataFrames with `pd.merge()` and `.join()`
- Diagnose and fix common merge problems (duplicate keys, mismatched types)

See `modules/03-pandas/notes.md` (Lesson 3.7) for the full written explanation.


In [1]:
import pandas as pd
from data_science_course.datasets import load_customers, load_orders, load_products

customers = load_customers()
orders = load_orders().drop_duplicates(subset=["order_id"])
products = load_products()
print(orders.shape, customers.shape)

(6004, 11) (800, 7)


`data/README.md` documents that 4 orders reference a `customer_id` that doesn't
exist in `customers.csv` (`C90001`-`C90004`), on purpose.

In [2]:
set(orders["customer_id"]) - set(customers["customer_id"])

{'C90001', 'C90002', 'C90003', 'C90004'}

## `how="inner"` vs `how="left"` on those orphan rows

In [3]:
inner = orders.merge(customers, on="customer_id", how="inner")
inner.shape

(6000, 17)

6,004 orders in, only 6,000 out -- the 4 orphan rows silently disappeared.

In [4]:
left = orders.merge(customers, on="customer_id", how="left")
left.shape

(6004, 17)

All 6,004 orders survive. Find the orphans by their `NaN` customer columns:

In [5]:
orphans = left[left["signup_date"].isna()]
orphans[["order_id", "customer_id", "product_id", "order_total", "signup_date", "region"]]

,order_id,customer_id,product_id,order_total,signup_date,region
56,O006003,C90003,P0044,81.68,NaN,NaN
2166,O006004,C90004,P0020,24.94,NaN,NaN
2329,O006001,C90001,P0043,39.70,NaN,NaN
4531,O006002,C90002,P0009,61.94,NaN,NaN


That's the real, concrete difference: `how="inner"` quietly drops rows you might not
know are gone (6,004 -> 6,000, no warning). `how="left"` keeps every row and turns the
problem into unmistakable `NaN`s you can find with `.isna()`. This is why `left` is
usually the safer default when merging *reference* data (like customer details) onto a
table you don't want to shrink (like orders).

## `how="right"` and `how="outer"` with `indicator=True`

In [6]:
right = customers.merge(orders, on="customer_id", how="right")
right.shape == left.shape   # same 6,004 rows -- just built from the other direction

True

In [7]:
outer = orders.merge(customers, on="customer_id", how="outer", indicator=True)
outer["_merge"].value_counts()

_merge
both          6000
right_only      69
left_only        4
Name: count, dtype: int64

One table, the whole story: 6,000 orders matched a real customer (`both`), 4 orders
are orphans (`left_only` -- in `orders` but not `customers`), and the rest of
`customers` (`right_only`) are customers who have never placed an order at all.
Running an outer merge with `indicator=True` is a great first move whenever you're
not sure how clean two tables' keys are relative to each other.

In [8]:
never_ordered = outer[outer["_merge"] == "right_only"]
never_ordered.shape[0]

69

## `.merge()` vs `.join()`

In [9]:
customers_by_id = customers.set_index("customer_id")
orders_by_id = orders.head(3).set_index("customer_id")

orders_by_id.join(customers_by_id, how="left", lsuffix="_ord")[["order_id", "region"]]

,order_id,region
customer_id,,
C00258,O000080,west
C00559,O002751,WEST
C00126,O003297,NORTH


`.merge()` matches on column values via `on=`. `.join()` matches on the **index** --
convenient when both tables are already indexed by the same key. `.merge()` is far
more common in practice.

## Diagnosing common merge problems

**Duplicate keys inflate row counts.** This is exactly why we deduped `orders` on
`order_id` before merging above -- a duplicate key merged against another table with
multiple matches would produce extra output rows and silently double-count revenue.

**A successful join doesn't guarantee the data means what you assumed.**
`shipping_region` usually matches the customer's `region`, but not always -- worth
checking directly rather than assuming a matched key means the values agree.

In [10]:
combined = orders.merge(customers, on="customer_id", how="inner")
combined["region_clean"] = combined["region"].str.strip().str.lower().str.title()
mismatch_rate = (combined["shipping_region"] != combined["region_clean"]).mean()
round(mismatch_rate, 3)

np.float64(0.112)

About 11% mismatch -- close to the ~10% `data/README.md` describes (gifts shipped
to a different address, etc.). Try the same comparison against the raw, un-cleaned
`region` column below -- you'll get a much higher, misleading mismatch rate, because
casing differences (`"west"` vs `"West"`) also count as "different." A merge key
matching correctly and a downstream *comparison* being meaningful are two separate
concerns.

In [11]:
raw_mismatch_rate = (combined["shipping_region"] != combined["region"]).mean()
round(raw_mismatch_rate, 3)

np.float64(0.623)

## Try it yourself

1. Merge `orders` with `products` (`how="left"`, on `product_id`) and confirm there
   are zero rows with a missing `product_name` (every `product_id` in `orders` should
   have a match, unlike `customer_id`).
2. Using the `outer` merge above, list the `customer_id` and `membership_tier` of the
   5 customers with the earliest `signup_date` among those who have never ordered
   (`_merge == "right_only"`).
3. Deliberately introduce a duplicate-key bug: merge `orders` (NOT deduplicated this
   time) with `customers` on `customer_id`, `how="inner"`, and compare `.shape[0]` to
   the deduplicated version above. Explain the difference in a comment.
4. Merge all three tables together (`orders` -> `customers` -> `products`) with
   `how="inner"` throughout, keeping only `order_id`, `region`, `category`, and
   `order_total`.


In [12]:
# 1. TODO


# 2. TODO


# 3. TODO


# 4. TODO


### Solution

In [13]:
# 1.
orders_products = orders.merge(products, on="product_id", how="left")
print(orders_products["product_name"].isna().sum())

# 2.
never_ordered_sorted = never_ordered.sort_values("signup_date").head(5)
print(never_ordered_sorted[["customer_id", "membership_tier", "signup_date"]])

# 3.
orders_raw = load_orders()  # NOT deduplicated -- still has the 5 duplicate order_id rows
inner_raw = orders_raw.merge(customers, on="customer_id", how="inner")
print(inner_raw.shape[0], inner.shape[0])
# inner_raw has 5 more rows than `inner` (the deduplicated version) -- each duplicate
# order_id row matches its customer again, double-counting that order.

# 4.
full = (
    orders.merge(customers, on="customer_id", how="inner")
    .merge(products, on="product_id", how="inner")
)[["order_id", "region", "category", "order_total"]]
print(full.shape)
print(full.head())

0
     customer_id membership_tier signup_date
1639      C00215            Gold  2022-02-09
3907      C00531          Silver  2022-02-10
297       C00028          Silver  2022-02-20
579       C00068          Bronze  2022-02-25
1223      C00171            Gold  2022-03-23
6005 6000
(6000, 4)
  order_id region           category  order_total
0  O000080   west           Clothing        72.48
1  O002751   WEST  Sports & Outdoors        69.18
2  O003297  NORTH     Home & Kitchen       201.03
3  O001169   East        Electronics       542.25
4  O003427  South        Electronics       486.05
